# Capítulo 10 — Regras de Associação: o que Costuma Aparecer Junto?

Códigos do capítulo. Execute as células em ordem; cada bloco é independente dos demais.


## 10.7 Mãos à obra com mlxtend

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

transacoes = [
    ["pão", "leite"],
    ["pão", "manteiga", "leite"],
    ["pão", "manteiga"],
    ["leite", "café"],
    ["pão", "leite", "café"],
    ["pão", "manteiga", "café"],
    ["leite", "manteiga"],
    ["pão", "manteiga", "leite", "café"],
    ["pão", "café"],
    ["pão", "leite", "manteiga"],
]

# 1) Transformar listas em tabela verdadeiro/falso (one-hot)
te = TransactionEncoder()
df = pd.DataFrame(te.fit(transacoes).transform(transacoes), columns=te.columns_)

# 2) Itemsets frequentes (suporte mínimo 0,3)
freq = apriori(df, min_support=0.3, use_colnames=True)
print(freq.sort_values("support", ascending=False))

# O FP-Growth devolve o mesmo resultado, normalmente mais rápido
freq2 = fpgrowth(df, min_support=0.3, use_colnames=True)
print(len(freq), len(freq2))

# 3) Regras de associação
regras = association_rules(freq, metric="confidence", min_threshold=0.6)
cols = ["antecedents", "consequents", "support", "confidence", "lift"]
print(regras[cols].sort_values("lift", ascending=False).round(2))

## 10.10 Estudo de caso: recomendação em uma loja virtual

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

pedidos = [
    ["notebook", "mouse", "mochila"], ["notebook", "mouse"], ["celular", "capinha", "película"],
    ["celular", "capinha"], ["notebook", "mochila"], ["celular", "película"],
    ["notebook", "mouse", "teclado"], ["celular", "capinha", "película", "carregador"],
    ["mouse", "teclado"], ["notebook", "mouse", "mochila", "teclado"],
] * 5

te = TransactionEncoder()
df = pd.DataFrame(te.fit(pedidos).transform(pedidos), columns=te.columns_)
freq = fpgrowth(df, min_support=0.15, use_colnames=True)
r = association_rules(freq, metric="lift", min_threshold=1.2)

def recomendar(produto, regras, n=3):
    sel = regras[regras["antecedents"].apply(lambda s: s == frozenset([produto]))]
    sel = sel.sort_values(["lift", "confidence"], ascending=False).head(n)
    return [list(c)[0] for c in sel["consequents"] if len(c) == 1]

print("Quem compra 'celular' também compra:", recomendar("celular", r))

## Resumo do capítulo